In [ ]:
# Cell 1: Environment setup
# Uncomment the next line in a fresh notebook runtime if dependencies are missing.
# %pip install -q torch matplotlib datasets

PROJECT_ROOT = "."
CHECKPOINT_DIR = "checkpoints"
RELEASE_DIR = "releases"
TOKENIZER_DIR = "tokenizer/terry_byte"
PROCESSED_DIR = "src/processed"
ENABLE_NOTEBOOK_PLOTS = True

print("Notebook environment setup complete.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")


In [ ]:
# Cell 2: Import all modules
from __future__ import annotations

import argparse
import json
import math
import os
import random
import time
import unittest
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Callable, Iterable, Iterator, Optional, Sequence

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, IterableDataset


In [ ]:
# Cell 3: General configuration
# Source: config.py
@dataclass
class ModelConfig:
    d_model: int = 256
    n_layers: int = 8
    n_heads: int = 8
    ffn_mult: int = 4
    max_seq_len: int = 8192  # Increased for longer contexts
    dropout_rate: float = 0.0
    # New parameters for efficient attention
    sliding_window: int = 2048  # Local attention window
    use_sliding_window: bool = True  # Enable sliding window attention


@dataclass
class TrainConfig:
    lr: float = 1e-4  # conservative for CPU small-batch training
    weight_decay: float = 0.01
    batch_size: int = 2  # Reduced for longer sequences
    grad_accum: int = 16  # Increased accumulation for effective batch size
    device: str = "auto"
    warmup_steps: int = 500
    total_steps: int = 10000
    seed: int = 42
    train_source_path: str = "src/fineweb_train.jsonl"
    valid_source_path: str = "src/fineweb_valid.jsonl"
    train_tokens_path: str = "src/processed/fineweb_train_tokens.txt"
    valid_tokens_path: str = "src/processed/fineweb_valid_tokens.txt"
    tokenizer_dir: str = "tokenizer/terry_byte"
    dataset_repo: str = "HuggingFaceFW/fineweb"
    dataset_name: str = "CC-MAIN-2025-26"
    dataset_split: str = "train"
    train_target_tokens: int = 500_000_000
    valid_target_tokens: int = 10_000_000
    buffer_size: int = 100_000
    language: str = "en"
    min_language_score: float = 0.95
    min_token_count: int = 100
    max_token_count: int = 4096
    use_mixed_precision: bool = True


# Source: tools/tokenizer.py
PAD_TOKEN = "<|pad|>"
BOS_TOKEN = "<|im_start|>"
EOS_TOKEN = "<|im_end|>"

PAD_TOKEN_ID = 0
BOS_TOKEN_ID = 1
EOS_TOKEN_ID = 2

BYTE_OFFSET = 3
VOCAB_SIZE = BYTE_OFFSET + 256

DEFAULT_TOKENIZER_DIR = Path("tokenizer/terry_byte")


class ByteTokenizer:
    """Tiny byte-level tokenizer with fixed chat special tokens."""

    def __init__(self, model_max_length: int | None = None):
        self.pad_token = PAD_TOKEN
        self.bos_token = BOS_TOKEN
        self.eos_token = EOS_TOKEN
        self.pad_token_id = PAD_TOKEN_ID
        self.bos_token_id = BOS_TOKEN_ID
        self.eos_token_id = EOS_TOKEN_ID
        self.model_max_length = model_max_length or max(
            ModelConfig().max_seq_len,
            1_000_000,
        )

    def __len__(self) -> int:
        return VOCAB_SIZE

    def encode(
        self,
        text: str,
        add_special_tokens: bool = False,
        return_tensors: str | None = None,
    ):
        if not isinstance(text, str):
            raise TypeError(f"Expected text to be str, got {type(text)!r}")

        ids = [BYTE_OFFSET + value for value in text.encode("utf-8")]

        if add_special_tokens:
            ids = [self.bos_token_id, *ids, self.eos_token_id]

        if return_tensors == "pt":
            return torch.tensor([ids], dtype=torch.long)

        return ids

    def decode(
        self,
        ids: Sequence[int] | torch.Tensor,
        skip_special_tokens: bool = False,
        clean_up_tokenization_spaces: bool = True,
    ) -> str:
        del clean_up_tokenization_spaces

        flat_ids = self._flatten_ids(ids)
        pieces: list[str] = []
        byte_buffer = bytearray()

        for token_id in flat_ids:
            if token_id >= BYTE_OFFSET:
                byte_buffer.append(token_id - BYTE_OFFSET)
                continue

            if byte_buffer:
                pieces.append(byte_buffer.decode("utf-8", errors="ignore"))
                byte_buffer.clear()

            if skip_special_tokens:
                continue

            if token_id == self.pad_token_id:
                pieces.append(self.pad_token)
            elif token_id == self.bos_token_id:
                pieces.append(self.bos_token)
            elif token_id == self.eos_token_id:
                pieces.append(self.eos_token)

        if byte_buffer:
            pieces.append(byte_buffer.decode("utf-8", errors="ignore"))

        return "".join(pieces)

    def __call__(
        self,
        text: str,
        add_special_tokens: bool = False,
        truncation: bool = False,
        return_tensors: str | None = None,
    ):
        del truncation
        input_ids = self.encode(
            text,
            add_special_tokens=add_special_tokens,
            return_tensors=return_tensors,
        )
        return SimpleNamespace(input_ids=input_ids)

    def convert_ids_to_tokens(self, ids: Iterable[int]) -> list[str]:
        tokens = []
        for token_id in ids:
            if token_id == self.pad_token_id:
                tokens.append(self.pad_token)
            elif token_id == self.bos_token_id:
                tokens.append(self.bos_token)
            elif token_id == self.eos_token_id:
                tokens.append(self.eos_token)
            else:
                tokens.append(
                    bytes([token_id - BYTE_OFFSET]).decode("utf-8", errors="ignore")
                )
        return tokens

    def save_pretrained(self, save_directory: str | Path):
        save_path = Path(save_directory)
        save_path.mkdir(parents=True, exist_ok=True)

        config = {
            "tokenizer_type": "byte",
            "pad_token": self.pad_token,
            "bos_token": self.bos_token,
            "eos_token": self.eos_token,
            "pad_token_id": self.pad_token_id,
            "bos_token_id": self.bos_token_id,
            "eos_token_id": self.eos_token_id,
            "byte_offset": BYTE_OFFSET,
            "vocab_size": VOCAB_SIZE,
            "model_max_length": self.model_max_length,
        }

        with (save_path / "tokenizer_config.json").open("w", encoding="utf-8") as handle:
            json.dump(config, handle, indent=2)

        return (str(save_path),)

    @classmethod
    def from_pretrained(cls, load_directory: str | Path) -> "ByteTokenizer":
        config_path = Path(load_directory) / "tokenizer_config.json"
        if not config_path.exists():
            raise FileNotFoundError(f"Tokenizer config not found at {config_path}")

        with config_path.open("r", encoding="utf-8") as handle:
            config = json.load(handle)

        tokenizer = cls(model_max_length=config.get("model_max_length"))
        expected = {
            "pad_token_id": PAD_TOKEN_ID,
            "bos_token_id": BOS_TOKEN_ID,
            "eos_token_id": EOS_TOKEN_ID,
        }
        for key, value in expected.items():
            if config.get(key) != value:
                raise ValueError(f"Unsupported tokenizer config: {key}={config.get(key)}")

        return tokenizer

    @staticmethod
    def _flatten_ids(ids: Sequence[int] | torch.Tensor) -> list[int]:
        if isinstance(ids, torch.Tensor):
            return ids.detach().cpu().view(-1).tolist()

        if ids and isinstance(ids[0], (list, tuple)):
            flat: list[int] = []
            for item in ids:
                flat.extend(item)
            return flat

        return list(ids)


def load_tokenizer(tokenizer_path: str | Path | None = None) -> ByteTokenizer:
    if tokenizer_path is not None:
        path = Path(tokenizer_path)
        if (path / "tokenizer_config.json").exists():
            return ByteTokenizer.from_pretrained(path)

    if (DEFAULT_TOKENIZER_DIR / "tokenizer_config.json").exists():
        return ByteTokenizer.from_pretrained(DEFAULT_TOKENIZER_DIR)

    return ByteTokenizer()


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def setup_device(device_cfg: str):
    if device_cfg == "auto":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device_cfg)

    print(f"Using device: {device}")

    if device.type == "cuda":
        torch.set_float32_matmul_precision("high")
    else:
        torch.set_num_threads(1)

    return device


In [ ]:
# Cell 4: Dataset builder / Prepare dataset

# FineWeb is streamed directly; the legacy Terry dataset generator was removed.

# Source: data/dataset_builder.py
# Internal project imports were flattened into earlier notebook cells.

DEFAULT_PROCESSED_DIR = Path("src/processed")
DEFAULT_TRAIN_SOURCE = Path("src/fineweb_train.jsonl")
DEFAULT_VALID_SOURCE = Path("src/fineweb_valid.jsonl")
DEFAULT_TRAIN_TOKENS = DEFAULT_PROCESSED_DIR / "fineweb_train_tokens.txt"
DEFAULT_VALID_TOKENS = DEFAULT_PROCESSED_DIR / "fineweb_valid_tokens.txt"
DEFAULT_TOKENIZER_DIR = Path("tokenizer/terry_byte")

DEFAULT_DATASET_REPO = "HuggingFaceFW/fineweb"
DEFAULT_DATASET_NAME = "CC-MAIN-2025-26"
DEFAULT_DATASET_SPLIT = "train"
DEFAULT_TRAIN_TARGET_TOKENS = 500_000_000
DEFAULT_VALID_TARGET_TOKENS = 10_000_000
DEFAULT_BUFFER_SIZE = 100_000
DEFAULT_LANGUAGE = "en"
DEFAULT_MIN_LANGUAGE_SCORE = 0.95
DEFAULT_MIN_TOKEN_COUNT = 100
DEFAULT_MAX_TOKEN_COUNT = 4096


def _load_hf_dataset():
    try:
        from datasets import load_dataset
    except ImportError as exc:
        raise ImportError(
            "The `datasets` package is required for FineWeb preparation. "
            "Install dependencies with `pip install -r requirements.txt`."
        ) from exc

    return load_dataset


def iter_jsonl_records(path: str | Path) -> Iterator[dict]:
    """Yield JSONL records from a dataset split."""
    file_path = Path(path)
    with file_path.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue
            yield json.loads(line)


def serialize_text_record(
    tokenizer,
    text: str,
) -> tuple[list[int], list[int]]:
    """Convert a plain text document into one trainable token sequence."""
    normalized = text.strip()
    if not normalized:
        return [], []

    content_tokens = tokenizer.encode(normalized, add_special_tokens=False)
    tokens = [tokenizer.bos_token_id, *content_tokens, tokenizer.eos_token_id]
    loss_mask = [0, *([1] * len(content_tokens)), 1]
    return tokens, loss_mask


def serialize_record(tokenizer, record: dict) -> tuple[list[int], list[int]]:
    """Serialize a plain-text dataset record."""
    if "text" in record:
        return serialize_text_record(tokenizer, record["text"])
    raise KeyError("Expected a record with a `text` field.")


def write_tokenized_split(
    source_path: str | Path,
    target_path: str | Path,
    tokenizer,
    max_seq_len: int = 8192,
) -> dict[str, int]:
    """Encode one JSONL split into one-doc-per-line token IDs with masks."""
    target = Path(target_path)
    mask_path = Path(str(target_path) + ".mask")
    target.parent.mkdir(parents=True, exist_ok=True)

    documents = 0
    total_tokens = 0

    with target.open("w", encoding="utf-8") as token_file, mask_path.open(
        "w", encoding="utf-8"
    ) as mask_file:
        for record in iter_jsonl_records(source_path):
            ids, mask = serialize_record(tokenizer, record)
            if len(ids) > max_seq_len:
                ids = ids[:max_seq_len]
                mask = mask[:max_seq_len]

            if len(ids) < 2:
                continue

            assert len(ids) == len(mask), (
                f"Token/mask mismatch: {len(ids)} tokens vs {len(mask)} mask values"
            )

            token_file.write(" ".join(map(str, ids)))
            token_file.write("\n")
            mask_file.write(" ".join(map(str, mask)))
            mask_file.write("\n")

            documents += 1
            total_tokens += len(ids)

    return {"documents": documents, "tokens": total_tokens}


def write_fineweb_splits(
    train_path: str | Path = DEFAULT_TRAIN_SOURCE,
    valid_path: str | Path = DEFAULT_VALID_SOURCE,
    dataset_repo: str = DEFAULT_DATASET_REPO,
    dataset_name: str = DEFAULT_DATASET_NAME,
    dataset_split: str = DEFAULT_DATASET_SPLIT,
    train_target_tokens: int = DEFAULT_TRAIN_TARGET_TOKENS,
    valid_target_tokens: int = DEFAULT_VALID_TARGET_TOKENS,
    buffer_size: int = DEFAULT_BUFFER_SIZE,
    language: str = DEFAULT_LANGUAGE,
    min_language_score: float = DEFAULT_MIN_LANGUAGE_SCORE,
    min_token_count: int = DEFAULT_MIN_TOKEN_COUNT,
    max_token_count: int = DEFAULT_MAX_TOKEN_COUNT,
    seed: int = 42,
) -> dict[str, object]:
    """Stream FineWeb, filter records, and cache train/valid JSONL splits."""
    load_dataset = _load_hf_dataset()

    train = Path(train_path)
    valid = Path(valid_path)
    train.parent.mkdir(parents=True, exist_ok=True)
    valid.parent.mkdir(parents=True, exist_ok=True)

    stream_ds = load_dataset(
        dataset_repo,
        name=dataset_name,
        split=dataset_split,
        streaming=True,
    )
    stream_ds = stream_ds.shuffle(seed=seed, buffer_size=buffer_size)

    rng = random.Random(seed)
    train_tokens = 0
    valid_tokens = 0
    train_samples = 0
    valid_samples = 0

    with train.open("w", encoding="utf-8") as train_handle, valid.open(
        "w", encoding="utf-8"
    ) as valid_handle:
        for sample in stream_ds:
            if sample.get("language") != language:
                continue

            language_score = float(sample.get("language_score") or 0.0)
            if language_score < min_language_score:
                continue

            token_count = int(sample.get("token_count") or 0)
            if token_count < min_token_count or token_count > max_token_count:
                continue

            text = str(sample.get("text") or "").strip()
            if not text:
                continue

            record = {
                "text": text,
                "language": language,
                "language_score": language_score,
                "token_count": token_count,
            }

            train_remaining = max(0, train_target_tokens - train_tokens)
            valid_remaining = max(0, valid_target_tokens - valid_tokens)
            if train_remaining == 0 and valid_remaining == 0:
                break

            if train_remaining > 0 and valid_remaining > 0:
                valid_probability = valid_remaining / (train_remaining + valid_remaining)
                write_to_valid = rng.random() < valid_probability
            else:
                write_to_valid = valid_remaining > 0

            handle = valid_handle if write_to_valid else train_handle
            handle.write(json.dumps(record, ensure_ascii=False))
            handle.write("\n")

            if write_to_valid:
                valid_tokens += token_count
                valid_samples += 1
            else:
                train_tokens += token_count
                train_samples += 1

            total_samples = train_samples + valid_samples
            if total_samples % 10_000 == 0:
                print(
                    f"samples={total_samples:,} | "
                    f"train_tokens={train_tokens:,} | "
                    f"valid_tokens={valid_tokens:,}"
                )

    return {
        "dataset_repo": dataset_repo,
        "dataset_name": dataset_name,
        "dataset_split": dataset_split,
        "train_path": str(train),
        "valid_path": str(valid),
        "train_samples": train_samples,
        "valid_samples": valid_samples,
        "train_source_tokens": train_tokens,
        "valid_source_tokens": valid_tokens,
    }


def ensure_source_dataset(
    train_path: str | Path = DEFAULT_TRAIN_SOURCE,
    valid_path: str | Path = DEFAULT_VALID_SOURCE,
    dataset_repo: str = DEFAULT_DATASET_REPO,
    dataset_name: str = DEFAULT_DATASET_NAME,
    dataset_split: str = DEFAULT_DATASET_SPLIT,
    train_target_tokens: int = DEFAULT_TRAIN_TARGET_TOKENS,
    valid_target_tokens: int = DEFAULT_VALID_TARGET_TOKENS,
    buffer_size: int = DEFAULT_BUFFER_SIZE,
    language: str = DEFAULT_LANGUAGE,
    min_language_score: float = DEFAULT_MIN_LANGUAGE_SCORE,
    min_token_count: int = DEFAULT_MIN_TOKEN_COUNT,
    max_token_count: int = DEFAULT_MAX_TOKEN_COUNT,
    seed: int = 42,
    force: bool = False,
) -> dict[str, object]:
    """Create cached FineWeb JSONL splits when they do not exist yet."""
    train = Path(train_path)
    valid = Path(valid_path)

    if force or not train.exists() or not valid.exists():
        return write_fineweb_splits(
            train_path=train,
            valid_path=valid,
            dataset_repo=dataset_repo,
            dataset_name=dataset_name,
            dataset_split=dataset_split,
            train_target_tokens=train_target_tokens,
            valid_target_tokens=valid_target_tokens,
            buffer_size=buffer_size,
            language=language,
            min_language_score=min_language_score,
            min_token_count=min_token_count,
            max_token_count=max_token_count,
            seed=seed,
        )

    return {
        "dataset_repo": dataset_repo,
        "dataset_name": dataset_name,
        "dataset_split": dataset_split,
        "train_path": str(train),
        "valid_path": str(valid),
        "train_samples": None,
        "valid_samples": None,
        "train_source_tokens": None,
        "valid_source_tokens": None,
    }


def prepare_dataset_assets(
    train_source: str | Path = DEFAULT_TRAIN_SOURCE,
    valid_source: str | Path = DEFAULT_VALID_SOURCE,
    train_tokens: str | Path = DEFAULT_TRAIN_TOKENS,
    valid_tokens: str | Path = DEFAULT_VALID_TOKENS,
    tokenizer_dir: str | Path = DEFAULT_TOKENIZER_DIR,
    dataset_repo: str = DEFAULT_DATASET_REPO,
    dataset_name: str = DEFAULT_DATASET_NAME,
    dataset_split: str = DEFAULT_DATASET_SPLIT,
    train_target_tokens: int = DEFAULT_TRAIN_TARGET_TOKENS,
    valid_target_tokens: int = DEFAULT_VALID_TARGET_TOKENS,
    buffer_size: int = DEFAULT_BUFFER_SIZE,
    language: str = DEFAULT_LANGUAGE,
    min_language_score: float = DEFAULT_MIN_LANGUAGE_SCORE,
    min_token_count: int = DEFAULT_MIN_TOKEN_COUNT,
    max_token_count: int = DEFAULT_MAX_TOKEN_COUNT,
    seed: int = 42,
    force: bool = False,
) -> dict[str, object]:
    """Ensure FineWeb source JSONL, tokenizer files, and tokenized splits exist."""
    source_stats = ensure_source_dataset(
        train_path=train_source,
        valid_path=valid_source,
        dataset_repo=dataset_repo,
        dataset_name=dataset_name,
        dataset_split=dataset_split,
        train_target_tokens=train_target_tokens,
        valid_target_tokens=valid_target_tokens,
        buffer_size=buffer_size,
        language=language,
        min_language_score=min_language_score,
        min_token_count=min_token_count,
        max_token_count=max_token_count,
        seed=seed,
        force=force,
    )

    tokenizer = load_tokenizer()
    tokenizer.save_pretrained(tokenizer_dir)

    train_stats = write_tokenized_split(
        train_source,
        train_tokens,
        tokenizer,
        max_seq_len=24576,
    )
    valid_stats = write_tokenized_split(
        valid_source,
        valid_tokens,
        tokenizer,
        max_seq_len=24576,
    )

    return {
        "dataset_repo": dataset_repo,
        "dataset_name": dataset_name,
        "dataset_split": dataset_split,
        "train_source": str(train_source),
        "valid_source": str(valid_source),
        "train_tokens": str(train_tokens),
        "valid_tokens": str(valid_tokens),
        "tokenizer_dir": str(tokenizer_dir),
        "train_documents": train_stats["documents"],
        "valid_documents": valid_stats["documents"],
        "train_token_count": train_stats["tokens"],
        "valid_token_count": valid_stats["tokens"],
        "train_source_samples": source_stats["train_samples"],
        "valid_source_samples": source_stats["valid_samples"],
        "train_source_token_count": source_stats["train_source_tokens"],
        "valid_source_token_count": source_stats["valid_source_tokens"],
    }

# Source: prepare_data.py
# Internal project imports were flattened into earlier notebook cells.

# Special tokens
pad_id: int = PAD_TOKEN_ID
bos_id: int = BOS_TOKEN_ID  # <|im_start|>
eos_id: int = EOS_TOKEN_ID  # <|im_end|>


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Stream FineWeb, cache filtered JSONL splits, and encode them for training.",
    )
    parser.add_argument("--train-source", type=Path, default=DEFAULT_TRAIN_SOURCE)
    parser.add_argument("--valid-source", type=Path, default=DEFAULT_VALID_SOURCE)
    parser.add_argument("--train-tokens", type=Path, default=DEFAULT_TRAIN_TOKENS)
    parser.add_argument("--valid-tokens", type=Path, default=DEFAULT_VALID_TOKENS)
    parser.add_argument("--tokenizer-dir", type=Path, default=DEFAULT_TOKENIZER_DIR)
    parser.add_argument("--dataset-repo", type=str, default=DEFAULT_DATASET_REPO)
    parser.add_argument("--dataset-name", type=str, default=DEFAULT_DATASET_NAME)
    parser.add_argument("--dataset-split", type=str, default=DEFAULT_DATASET_SPLIT)
    parser.add_argument(
        "--train-target-tokens",
        type=int,
        default=DEFAULT_TRAIN_TARGET_TOKENS,
    )
    parser.add_argument(
        "--valid-target-tokens",
        type=int,
        default=DEFAULT_VALID_TARGET_TOKENS,
    )
    parser.add_argument("--buffer-size", type=int, default=DEFAULT_BUFFER_SIZE)
    parser.add_argument("--language", type=str, default=DEFAULT_LANGUAGE)
    parser.add_argument(
        "--min-language-score",
        type=float,
        default=DEFAULT_MIN_LANGUAGE_SCORE,
    )
    parser.add_argument("--min-token-count", type=int, default=DEFAULT_MIN_TOKEN_COUNT)
    parser.add_argument("--max-token-count", type=int, default=DEFAULT_MAX_TOKEN_COUNT)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--force",
        action="store_true",
        help="Regenerate the cached FineWeb JSONL splits before tokenization.",
    )
    args, _ = parser.parse_known_args()
    return args


def main():
    args = parse_args()
    stats = prepare_dataset_assets(
        train_source=args.train_source,
        valid_source=args.valid_source,
        train_tokens=args.train_tokens,
        valid_tokens=args.valid_tokens,
        tokenizer_dir=args.tokenizer_dir,
        dataset_repo=args.dataset_repo,
        dataset_name=args.dataset_name,
        dataset_split=args.dataset_split,
        train_target_tokens=args.train_target_tokens,
        valid_target_tokens=args.valid_target_tokens,
        buffer_size=args.buffer_size,
        language=args.language,
        min_language_score=args.min_language_score,
        min_token_count=args.min_token_count,
        max_token_count=args.max_token_count,
        seed=args.seed,
        force=args.force,
    )

    print("Prepared FineWeb dataset assets")
    print(f"pad_id={pad_id} bos_id={bos_id} eos_id={eos_id}")
    print(
        "dataset: "
        f"{stats['dataset_repo']} / {stats['dataset_name']} [{stats['dataset_split']}]"
    )
    print(f"train source: {stats['train_source']}")
    print(f"valid source: {stats['valid_source']}")
    if stats["train_source_token_count"] is None:
        print("source totals: reused cached FineWeb splits")
    else:
        print(
            "source totals: "
            f"train={stats['train_source_token_count']} "
            f"valid={stats['valid_source_token_count']}"
        )
    print(f"train tokens: {stats['train_tokens']} ({stats['train_documents']} docs)")
    print(f"valid tokens: {stats['valid_tokens']} ({stats['valid_documents']} docs)")
    print(f"tokenizer dir: {stats['tokenizer_dir']}")


if __name__ == "__main__":
    main()

# Source: tools/converter.py
# Internal project imports were flattened into earlier notebook cells.

def pack_text_files(
    input_paths,
    output_path="data/packed_tokens.txt",
    add_eos=True,
    min_chars=1,
):
    """
    Converts raw text files into a packed token ID file.

    Args:
        input_paths (list[str]): Paths to .txt files with raw text
        output_path (str): Where to write packed token IDs
        add_eos (bool): Append EOS token after each document
        min_chars (int): Skip very short documents
    """
    tokenizer = load_tokenizer()
    eos_id = tokenizer.eos_token_id

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    total_tokens = 0
    doc_count = 0

    with open(output_path, "w", encoding="utf-8") as out:
        for path in input_paths:
            path = Path(path)
            if not path.exists():
                raise FileNotFoundError(path)

            with open(path, "r", encoding="utf-8") as f:
                text = f.read().strip()

            if len(text) < min_chars:
                continue

            # Tokenize WITHOUT adding special tokens
            ids = tokenizer(
                text,
                add_special_tokens=False,
                truncation=False,
            ).input_ids

            if not ids:
                continue

            if add_eos:
                ids.append(eos_id)

            # Write as space-separated integers
            out.write(" ".join(map(str, ids)))
            out.write("\n")

            total_tokens += len(ids)
            doc_count += 1

    print(f"Packed {doc_count} documents")
    print(f"Total tokens: {total_tokens}")
    print(f"Output written to: {output_path}")


In [ ]:
# Cell 5: Dataset loader

# Source: data/dataset.py
# Internal project imports were flattened into earlier notebook cells.

class TokenDataset(Dataset):
    """
    Document-aware causal language modeling dataset with strided windows.

    Emitted samples never exceed ``max_seq_len``, but long documents are kept
    intact so we can produce multiple training windows instead of silently
    truncating them at load time.
    """

    def __init__(self, documents, max_seq_len, min_seq_len=32, stride=None):
        self.max_seq_len = max_seq_len
        self.min_seq_len = min_seq_len
        self.stride = stride or min_seq_len

        self.documents = [
            doc for doc in documents if len(doc) >= min_seq_len + 1
        ]

        self.samples = []
        for doc_idx, doc in enumerate(self.documents):
            max_start = len(doc) - min_seq_len - 1
            if max_start <= 0:
                continue

            for start_idx in range(0, max_start, self.stride):
                self.samples.append((doc_idx, start_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        doc_idx, start_idx = self.samples[idx]
        doc = self.documents[doc_idx]

        # Remaining tokens available for causal shift
        remaining = len(doc) - start_idx - 1
        seq_len = min(self.max_seq_len, remaining)

        x = doc[start_idx : start_idx + seq_len]
        y = doc[start_idx + 1 : start_idx + seq_len + 1]

        # Absolute safety cap (never exceed max_seq_len)
        assert len(x) <= self.max_seq_len

        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(y, dtype=torch.long),
        )


# Source: data/stream_dataset.py
# Internal project imports were flattened into earlier notebook cells.

class StreamingTokenDataset(IterableDataset):
    """
    Stream tokenized documents from disk and yield causal LM samples with loss masks.

    Each line is one full document encoded as space-separated token IDs.
    Optionally loads corresponding loss masks (1=train, 0=ignore).
    """

    def __init__(
        self,
        path: str,
        max_seq_len: int,
        min_seq_len: int = 32,
        stride: Optional[int] = None,
    ):
        self.path = path
        self.mask_path = Path(str(path) + ".mask")
        self.has_masks = self.mask_path.exists()

        self.max_seq_len = max_seq_len
        self.min_seq_len = min_seq_len
        self.stride = stride or min_seq_len

    def _doc_to_samples(self, doc_tokens, doc_masks=None):
        """Generate samples from a document with optional loss mask.

        Args:
            doc_tokens: list of token IDs
            doc_masks: list of loss mask values (0=ignore, 1=train) or None

        Yields:
            Tuple of (x, y, loss_mask) tensors
            x and y are shifted by 1 for causal LM
            loss_mask[i] indicates whether y[i] should contribute to loss
        """
        n = len(doc_tokens)
        if n < self.min_seq_len + 1:
            return

        max_start = n - self.min_seq_len - 1
        for start in range(0, max_start + 1, self.stride):
            remaining = n - start - 1
            seq_len = min(self.max_seq_len, remaining)

            x = doc_tokens[start : start + seq_len]
            y = doc_tokens[start + 1 : start + seq_len + 1]

            # Extract corresponding loss mask
            if doc_masks:
                # Mask is shifted by 1 to align with y prediction targets
                loss_mask = doc_masks[start + 1 : start + seq_len + 1]
            else:
                # No mask: train on all tokens
                loss_mask = [1] * len(y)

            yield (
                torch.tensor(x, dtype=torch.long),
                torch.tensor(y, dtype=torch.long),
                torch.tensor(loss_mask, dtype=torch.long),
            )

    def parse_line(self, line: str):
        """Parse space-separated integers from a line."""
        try:
            toks = [int(x) for x in line.strip().split() if x]
            return toks
        except Exception:
            return []

    def __iter__(self):
        """Iterate documents and yield samples with loss masks."""
        if self.has_masks:
            # Load both tokens and masks in parallel
            with open(self.path, "r", encoding="utf-8") as token_file, \
                 open(self.mask_path, "r", encoding="utf-8") as mask_file:
                for token_line, mask_line in zip(token_file, mask_file):
                    doc_tokens = self.parse_line(token_line)
                    doc_masks = self.parse_line(mask_line)

                    if not doc_tokens or not doc_masks:
                        continue

                    # Sanity check
                    if len(doc_tokens) != len(doc_masks):
                        print(f"Warning: token/mask mismatch: {len(doc_tokens)} vs {len(doc_masks)}")
                        continue

                    for sample in self._doc_to_samples(doc_tokens, doc_masks):
                        yield sample
        else:
            # Legacy mode: no masks available
            with open(self.path, "r", encoding="utf-8") as f:
                for line in f:
                    doc = self.parse_line(line)
                    if not doc:
                        continue
                    for sample in self._doc_to_samples(doc):
                        yield sample


# Source: data/pipeline.py
# Internal project imports were flattened into earlier notebook cells.

def build_collate_fn(tokenizer):
    """Pad each batch to the longest sequence, handling loss masks."""
    pad_id = tokenizer.pad_token_id

    def collate_fn(batch):
        # Handle both old (x, y) and new (x, y, mask) formats
        if len(batch[0]) == 3:
            input_seqs, target_seqs, loss_masks = zip(*batch)
            has_masks = True
        else:
            input_seqs, target_seqs = zip(*batch)
            loss_masks = None
            has_masks = False

        max_len = max(seq.size(0) for seq in input_seqs)

        x = torch.stack(
            [
                F.pad(seq, (0, max_len - seq.size(0)), value=pad_id)
                for seq in input_seqs
            ]
        )
        y = torch.stack(
            [
                F.pad(seq, (0, max_len - seq.size(0)), value=pad_id)
                for seq in target_seqs
            ]
        )

        if has_masks:
            # Pad loss masks with 0 (don't train on padding)
            loss_mask = torch.stack(
                [
                    F.pad(mask.unsqueeze(0), (0, max_len - mask.size(0)), value=0).squeeze(0)
                    for mask in loss_masks
                ]
            )
            return x, y, loss_mask

        return x, y

    return collate_fn


def build_dataloader(tokenizer, model_cfg, train_cfg, use_cuda):
    """Build the FineWeb training dataloader.

    If the dataset or tokenized assets are missing, they are streamed and cached locally.
    """
    train_tokens = Path(train_cfg.train_tokens_path)
    valid_tokens = Path(train_cfg.valid_tokens_path)
    tokenizer_dir = Path(train_cfg.tokenizer_dir)

    if not train_tokens.exists() or not valid_tokens.exists() or not tokenizer_dir.exists():
        prepare_dataset_assets(
            train_source=train_cfg.train_source_path,
            valid_source=train_cfg.valid_source_path,
            train_tokens=train_cfg.train_tokens_path,
            valid_tokens=train_cfg.valid_tokens_path,
            tokenizer_dir=train_cfg.tokenizer_dir,
            dataset_repo=train_cfg.dataset_repo,
            dataset_name=train_cfg.dataset_name,
            dataset_split=train_cfg.dataset_split,
            train_target_tokens=train_cfg.train_target_tokens,
            valid_target_tokens=train_cfg.valid_target_tokens,
            buffer_size=train_cfg.buffer_size,
            language=train_cfg.language,
            min_language_score=train_cfg.min_language_score,
            min_token_count=train_cfg.min_token_count,
            max_token_count=train_cfg.max_token_count,
            seed=train_cfg.seed,
            force=False,
        )

    dataset = StreamingTokenDataset(
        path=train_cfg.train_tokens_path,
        max_seq_len=model_cfg.max_seq_len,
        min_seq_len=32,
    )

    loader = DataLoader(
        dataset,
        batch_size=train_cfg.batch_size,
        shuffle=not isinstance(dataset, IterableDataset),
        drop_last=not isinstance(dataset, IterableDataset),
        pin_memory=use_cuda,
        num_workers=0,
        collate_fn=build_collate_fn(tokenizer),
    )

    return loader


def build_validation_dataloader(tokenizer, model_cfg, train_cfg, use_cuda):
    """Notebook helper for building the validation dataloader."""
    train_tokens = Path(train_cfg.train_tokens_path)
    valid_tokens = Path(train_cfg.valid_tokens_path)
    tokenizer_dir = Path(train_cfg.tokenizer_dir)

    if not train_tokens.exists() or not valid_tokens.exists() or not tokenizer_dir.exists():
        prepare_dataset_assets(
            train_source=train_cfg.train_source_path,
            valid_source=train_cfg.valid_source_path,
            train_tokens=train_cfg.train_tokens_path,
            valid_tokens=train_cfg.valid_tokens_path,
            tokenizer_dir=train_cfg.tokenizer_dir,
            dataset_repo=train_cfg.dataset_repo,
            dataset_name=train_cfg.dataset_name,
            dataset_split=train_cfg.dataset_split,
            train_target_tokens=train_cfg.train_target_tokens,
            valid_target_tokens=train_cfg.valid_target_tokens,
            buffer_size=train_cfg.buffer_size,
            language=train_cfg.language,
            min_language_score=train_cfg.min_language_score,
            min_token_count=train_cfg.min_token_count,
            max_token_count=train_cfg.max_token_count,
            seed=train_cfg.seed,
            force=False,
        )

    dataset = StreamingTokenDataset(
        path=train_cfg.valid_tokens_path,
        max_seq_len=model_cfg.max_seq_len,
        min_seq_len=32,
    )

    return DataLoader(
        dataset,
        batch_size=train_cfg.batch_size,
        shuffle=False,
        drop_last=False,
        pin_memory=use_cuda,
        num_workers=0,
        collate_fn=build_collate_fn(tokenizer),
    )


In [ ]:
# Cell 6: Training (log and visual the status numbers in graphs)

# Source: model/rope.py
# Internal project imports were flattened into earlier notebook cells.

def build_rope_cache(seq_len, head_dim, device):
    """
    Build RoPE cache for rotary embeddings.

    Args:
        seq_len: Maximum sequence length
        head_dim: Dimension of each attention head
        device: Device to place the cache on

    Returns:
        Tuple of (cos, sin) tensors
    """
    # Standard RoPE formula: theta_i = 10000^(-2i/head_dim)
    # This is equivalent to: inv_freq = 1 / (10000^(2i/head_dim))
    inv_freq = 1.0 / (
        10000 ** (torch.arange(0, head_dim, 2, device=device).float() * 2 / head_dim)
    )
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """
    Apply RoPE to input tensor.

    Args:
        x: (B, H, T, D)
        cos: (T, D//2)
        sin: (T, D//2)
    """
    # Expand for broadcasting
    cos = cos.unsqueeze(0).unsqueeze(0)  # (1, 1, T, D//2)
    sin = sin.unsqueeze(0).unsqueeze(0)

    x1 = x[..., ::2]
    x2 = x[..., 1::2]

    out = torch.empty_like(x)
    out[..., ::2] = x1 * cos - x2 * sin
    out[..., 1::2] = x1 * sin + x2 * cos
    return out


# Source: model/norm.py
# Internal project imports were flattened into earlier notebook cells.

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps


    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return self.weight * x * torch.rsqrt(norm + self.eps)


# Source: model/ffn.py
# Internal project imports were flattened into earlier notebook cells.

class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w2 = nn.Linear(d_model, hidden, bias=False)
        self.w3 = nn.Linear(hidden, d_model, bias=False)


    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


# Source: model/attention.py
# Internal project imports were flattened into earlier notebook cells.

class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len=8192, sliding_window=2048, use_sliding_window=True):
        super().__init__()

        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.max_seq_len = max_seq_len
        self.sliding_window = sliding_window
        self.use_sliding_window = use_sliding_window

        self.qkv = nn.Linear(d_model, d_model * 3, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

        self.register_buffer("cos", None, persistent=False)
        self.register_buffer("sin", None, persistent=False)
        self.register_buffer("causal_mask", None, persistent=False)

    def _build_sliding_window_mask(self, max_len, device):
        """Build sliding window attention mask with local window + global attention."""
        # Create base causal mask
        causal_mask = torch.tril(torch.ones(max_len, max_len, device=device)).bool()

        if not self.use_sliding_window:
            return causal_mask

        # Apply sliding window constraint
        window_mask = torch.zeros(max_len, max_len, device=device, dtype=torch.bool)

        for i in range(max_len):
            # Each position can attend to:
            # 1. Previous positions within the sliding window
            start = max(0, i - self.sliding_window + 1)
            window_mask[i, start:i+1] = True

            # 2. Global attention to key positions (every N tokens)
            # For simplicity, allow attention to positions that are multiples of sliding_window/4
            global_step = self.sliding_window // 4
            for j in range(0, i+1, global_step):
                window_mask[i, j] = True

        # Combine causal and sliding window masks
        return causal_mask & window_mask

    def forward(self, x, padding_mask=None):
        """
        Args:
            x: Tensor of shape (B, T, C).
            padding_mask: Optional boolean tensor of shape (B, T).
                ``True`` marks valid tokens and ``False`` marks padding.
        """
        batch_size, seq_len, channels = x.size()
        device = x.device

        if self.cos is None or self.cos.device != device:
            self.cos, self.sin = build_rope_cache(
                self.max_seq_len,
                self.head_dim,
                device,
            )

        if self.causal_mask is None or self.causal_mask.device != device:
            if self.use_sliding_window:
                # Create sliding window causal mask
                self.causal_mask = self._build_sliding_window_mask(self.max_seq_len, device)
            else:
                self.causal_mask = torch.tril(
                    torch.ones(self.max_seq_len, self.max_seq_len, device=device)
                ).bool()

        causal_mask = self.causal_mask[:seq_len, :seq_len]

        qkv = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [
            tensor.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
            for tensor in qkv
        ]

        q = apply_rope(q, self.cos[:seq_len], self.sin[:seq_len])
        k = apply_rope(k, self.cos[:seq_len], self.sin[:seq_len])

        attn = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = attn.masked_fill(
            ~causal_mask.unsqueeze(0).unsqueeze(0),
            float("-inf"),
        )

        if padding_mask is not None:
            key_padding_mask = padding_mask.unsqueeze(1).unsqueeze(2)
            query_padding_mask = padding_mask.unsqueeze(1).unsqueeze(-1)
            attn = attn.masked_fill(~key_padding_mask, float("-inf"))
            attn = attn.masked_fill(~query_padding_mask, float("-inf"))

        attn = torch.softmax(attn, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, channels)

        if padding_mask is not None:
            out = out * padding_mask.unsqueeze(-1)

        return self.out(out)


# Source: model/transformer.py
# Internal project imports were flattened into earlier notebook cells.

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = RMSNorm(config.d_model)
        self.attn = SelfAttention(
            config.d_model,
            config.n_heads,
            config.max_seq_len,
            config.sliding_window,
            config.use_sliding_window,
        )
        self.norm2 = RMSNorm(config.d_model)
        self.ffn = SwiGLU(
            config.d_model,
            config.d_model * config.ffn_mult,
        )
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(self, x, padding_mask=None):
        """
        Args:
            x: (B, T, C)
            padding_mask: (B, T) boolean, True = valid token
        """
        attn_out = self.attn(self.norm1(x), padding_mask=padding_mask)
        x = x + self.dropout(attn_out)

        ffn_out = self.ffn(self.norm2(x))
        x = x + self.dropout(ffn_out)
        return x


class TinyLLM(nn.Module):
    def __init__(self, config, vocab_size: int):
        super().__init__()
        self.config = config
        self.config.vocab_size = vocab_size

        self.embed = nn.Embedding(self.config.vocab_size, config.d_model)

        self.blocks = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config.n_layers)]
        )

        self.norm = RMSNorm(config.d_model)
        self.head = nn.Linear(config.d_model, self.config.vocab_size, bias=False)

        # Weight tying
        self.head.weight = self.embed.weight

        self.gradient_checkpointing = False

    def gradient_checkpointing_enable(self):
        self.gradient_checkpointing = True

    def gradient_checkpointing_disable(self):
        self.gradient_checkpointing = False

    def resize_token_embeddings(self, new_vocab_size: int):
        self.config.vocab_size = new_vocab_size

        new_embed = nn.Embedding(self.config.vocab_size, self.config.d_model)
        new_head = nn.Linear(self.config.d_model, self.config.vocab_size, bias=False)

        # Copy old weights
        n = min(self.embed.weight.shape[0], new_embed.weight.shape[0])
        new_embed.weight.data[:n] = self.embed.weight.data[:n]
        new_head.weight.data[:n] = self.head.weight.data[:n]

        self.embed = new_embed
        self.head = new_head

        # Weight tying
        self.head.weight = self.embed.weight

    def forward(self, x, padding_mask=None):
        """
        Args:
            x: (B, T)
            padding_mask: (B, T) boolean
        """
        x = self.embed(x)

        if self.gradient_checkpointing and self.training:
            # Use gradient checkpointing for memory efficiency

            def create_custom_forward(module):
                def custom_forward(*inputs):
                    return module(*inputs)
                return custom_forward

            for block in self.blocks:
                x = checkpoint.checkpoint(
                    create_custom_forward(block),
                    x,
                    padding_mask,
                    use_reentrant=False,
                )
        else:
            for block in self.blocks:
                x = block(x, padding_mask=padding_mask)

        x = self.norm(x)
        return self.head(x)

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_length=100,
        temperature=1.0,
        top_k=50,
        top_p=0.9,
        do_sample=True,
        pad_token_id=None,
        eos_token_id=None,
    ):

        if pad_token_id is None:
            raise ValueError("pad_token_id must be provided")
        if eos_token_id is None:
            raise ValueError("eos_token_id must be provided")
        if max_length <= 0:
            raise ValueError("max_length must be positive")

        device = input_ids.device
        batch_size, seq_len = input_ids.shape
        temperature = max(temperature, 1e-5)

        if seq_len >= max_length:
            return input_ids[:, -max_length:].clone()

        generated = torch.full(
            (batch_size, max_length),
            pad_token_id,
            dtype=torch.long,
            device=device,
        )
        generated[:, :seq_len] = input_ids
        finished_sequences = torch.zeros(
            batch_size, dtype=torch.bool, device=device
        )

        for cur_len in range(seq_len, max_length):
            # Respect model context window
            start_idx = max(0, cur_len - self.config.max_seq_len)
            input_slice = generated[:, start_idx:cur_len]

            padding_mask = input_slice != pad_token_id

            logits = self(
                input_slice,
                padding_mask=padding_mask,
            )
            next_token_logits = logits[:, -1, :] / temperature

            if do_sample:
                # Top-p (nucleus) filtering
                if top_p < 1.0:
                    next_token_logits = self._top_p_filter(next_token_logits, top_p)

                # Top-k filtering
                if top_k > 0:
                    next_token_logits = self._top_k_filter(next_token_logits, top_k)

                probs = F.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = torch.argmax(
                    next_token_logits, dim=-1, keepdim=True
                )

            # Do not update finished sequences
            next_token[finished_sequences] = pad_token_id
            generated[:, cur_len] = next_token.squeeze(-1)

            # Mark finished sequences
            finished_sequences |= next_token.squeeze(-1) == eos_token_id

            # Stop if all sequences are finished
            if torch.all(finished_sequences):
                break

        return generated

    def _top_p_filter(self, logits, top_p):

        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

        # Remove tokens with cumulative probability above the threshold
        sorted_indices_to_remove = cum_probs > top_p
        # Shift the indices to the right to keep also the first token above the threshold
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[
            ..., :-1
        ].clone()
        sorted_indices_to_remove[..., 0] = 0

        indices_to_remove = sorted_indices_to_remove.scatter(
            1, sorted_indices, sorted_indices_to_remove
        )
        logits[indices_to_remove] = float("-inf")
        return logits

    def _top_k_filter(self, logits, top_k):
        top_k = min(top_k, logits.size(-1))  # Safety check
        # Remove all tokens with a probability less than the last token of the top-k
        indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
        logits[indices_to_remove] = float("-inf")
        return logits


# Source: utils/checkpoint.py
# Internal project imports were flattened into earlier notebook cells.

def save_checkpoint(model, optimizer, step, path, scheduler=None):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    checkpoint = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "step": step,
    }
    if scheduler is not None:
        checkpoint["scheduler"] = scheduler.state_dict()
    torch.save(checkpoint, path)


# Source: utils/logger.py
# Internal project imports were flattened into earlier notebook cells.

class Logger:
    def __init__(self, window_size=100):
        self.start = time.time()
        self.window_size = window_size
        self.losses = deque(maxlen=window_size)
        self.total_steps = 0
        self.total_loss = 0.0

    def log(self, step, loss, lr=None):
        # Update running statistics
        self.losses.append(loss)
        self.total_steps += 1
        self.total_loss += loss

        elapsed = time.time() - self.start
        avg_loss = sum(self.losses) / len(self.losses)
        total_avg_loss = self.total_loss / self.total_steps

        # Format learning rate if provided
        lr_str = f" lr={lr:.2e}" if lr is not None else ""

        # Enhanced logging with better formatting
        print(
            f"[{elapsed:7.1f}s] step={step:6d} "
            f"loss={loss:.4f} avg_loss={avg_loss:.4f} total_avg={total_avg_loss:.4f}{lr_str}"
        )


# Source: engine/trainer.py
# Internal project imports were flattened into earlier notebook cells.

class Trainer:
    def __init__(
        self,
        model,
        optimizer,
        scheduler,
        tokenizer,
        train_cfg,
        device,
        logger,
        checkpoint_path="checkpoints/last_model.pt",
        scaler=None,
    ):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.tokenizer = tokenizer
        self.train_cfg = train_cfg
        self.device = device
        self.logger = logger
        self.checkpoint_path = checkpoint_path
        self.scaler = scaler

        self.optimizer_step = 0
        self.micro_step = 0
        self.accum_loss = 0.0

    # --------------------------------------------------------------
    # Resume
    # --------------------------------------------------------------
    def try_resume(self):

        if not os.path.exists(self.checkpoint_path):
            return

        print(f"Loading checkpoint from {self.checkpoint_path}")
        checkpoint = torch.load(self.checkpoint_path, map_location=self.device)

        self.model.load_state_dict(checkpoint["model"])
        self.optimizer.load_state_dict(checkpoint["optimizer"])
        self.optimizer_step = checkpoint["step"]

        if "scheduler" in checkpoint:
            self.scheduler.load_state_dict(checkpoint["scheduler"])

        print(f"Resumed from step {self.optimizer_step}")

    # --------------------------------------------------------------
    # Step
    # --------------------------------------------------------------
    def train_step(self, x, y, loss_mask=None):
        """Train step with optional loss mask for assistant-only tokens.

        Args:
            x: input token IDs
            y: target token IDs (shifted by 1)
            loss_mask: optional binary mask (1=train, 0=ignore) for each position in y
        """
        x, y = x.to(self.device), y.to(self.device)
        if loss_mask is not None:
            loss_mask = loss_mask.to(self.device)

        if self.scaler is not None:
            # Mixed precision training
            with torch.amp.autocast(device_type=self.device.type, dtype=torch.float16):
                padding_mask = x != self.tokenizer.pad_token_id
                logits = self.model(x, padding_mask=padding_mask)

                # Reshape for cross entropy
                logits_flat = logits.reshape(-1, logits.size(-1))
                y_flat = y.reshape(-1)

                # Compute loss with padding ignored
                loss = F.cross_entropy(
                    logits_flat,
                    y_flat,
                    ignore_index=self.tokenizer.pad_token_id,
                    reduction="none",
                )

                # Apply loss mask if provided (only train on assistant tokens)
                if loss_mask is not None:
                    loss_mask_flat = loss_mask.reshape(-1).float()
                    valid_mask = (y_flat != self.tokenizer.pad_token_id)
                    effective_mask = loss_mask_flat * valid_mask.float()

                    loss = loss * effective_mask
                    tokens = effective_mask.sum()
                else:
                    # Legacy: count non-padding tokens
                    tokens = (y_flat != self.tokenizer.pad_token_id).sum()

                # Sum loss and normalize
                if tokens > 0:
                    loss = loss.sum() / tokens / self.train_cfg.grad_accum
                    self.scaler.scale(loss).backward()
                    self.accum_loss += loss.item()
                self.micro_step += 1
        else:
            # Standard precision training
            padding_mask = x != self.tokenizer.pad_token_id
            logits = self.model(x, padding_mask=padding_mask)

            # Reshape for cross entropy
            logits_flat = logits.reshape(-1, logits.size(-1))
            y_flat = y.reshape(-1)

            # Compute loss with padding ignored
            loss = F.cross_entropy(
                logits_flat,
                y_flat,
                ignore_index=self.tokenizer.pad_token_id,
                reduction="none",
            )

            # Apply loss mask if provided (only train on assistant tokens)
            if loss_mask is not None:
                loss_mask_flat = loss_mask.reshape(-1).float()
                valid_mask = (y_flat != self.tokenizer.pad_token_id)
                effective_mask = loss_mask_flat * valid_mask.float()

                loss = loss * effective_mask
                tokens = effective_mask.sum()
            else:
                # Legacy: count non-padding tokens
                tokens = (y_flat != self.tokenizer.pad_token_id).sum()

            # Sum loss and normalize
            if tokens > 0:
                loss = loss.sum() / tokens / self.train_cfg.grad_accum
                loss.backward()
                self.accum_loss += loss.item()
            self.micro_step += 1

    # --------------------------------------------------------------
    # Optimizer step
    # --------------------------------------------------------------
    def optimizer_step_fn(self):
        if self.scaler is not None:
            # Mixed precision step
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            # Standard precision step
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()

        self.scheduler.step()
        self.optimizer.zero_grad(set_to_none=True)

        self.optimizer_step += 1

    # --------------------------------------------------------------
    # Logging + checkpoint
    # --------------------------------------------------------------
    def maybe_log_and_save(self):
        if self.optimizer_step % 50 != 0:
            return

        avg_loss = self.accum_loss / 50
        self.accum_loss = 0.0

        save_checkpoint(
            model=self.model,
            optimizer=self.optimizer,
            step=self.optimizer_step,
            scheduler=self.scheduler,
            path=self.checkpoint_path,
        )

        self.tokenizer.save_pretrained("checkpoints/tokenizer")

        self.logger.log(
            step=self.optimizer_step,
            loss=avg_loss,
            lr=self.optimizer.param_groups[0]["lr"],
        )

    # --------------------------------------------------------------
    # Train loop
    # --------------------------------------------------------------
    def train(self, dataloader):
        self.model.train()

        while self.optimizer_step < self.train_cfg.total_steps:
            for batch in dataloader:
                if self.optimizer_step >= self.train_cfg.total_steps:
                    break

                # Handle both old (x, y) and new (x, y, mask) formats
                if len(batch) == 3:
                    x, y, loss_mask = batch
                    self.train_step(x, y, loss_mask)
                else:
                    x, y = batch
                    self.train_step(x, y)

                if self.micro_step % self.train_cfg.grad_accum == 0:
                    self.optimizer_step_fn()
                    self.maybe_log_and_save()


# Source: train.py
# Internal project imports were flattened into earlier notebook cells.

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def setup_device(device_cfg: str):
    if device_cfg == "auto":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device_cfg)

    print(f"Using device: {device}")

    if device.type == "cuda":
        torch.set_float32_matmul_precision("high")
    else:
        torch.set_num_threads(1)

    return device


def train_main():
    model_cfg = ModelConfig()
    train_cfg = TrainConfig()

    set_seed(train_cfg.seed)
    device = setup_device(train_cfg.device)
    use_cuda = device.type == "cuda"

    tokenizer = load_tokenizer()

    # Data
    loader = build_dataloader(tokenizer, model_cfg, train_cfg, use_cuda)

    # Model
    model = TinyLLM(model_cfg, vocab_size=len(tokenizer)).to(device)

    # Enable gradient checkpointing for memory efficiency with long sequences
    if model_cfg.max_seq_len > 4096:
        model.gradient_checkpointing_enable()

    # Mixed precision training
    scaler = None
    if train_cfg.use_mixed_precision and device.type == "cuda":
        scaler = torch.amp.GradScaler()
        print("Using mixed precision training")

    optimizer = AdamW(
        model.parameters(),
        lr=train_cfg.lr,
        weight_decay=train_cfg.weight_decay,
        fused=use_cuda,
    )

    def lr_lambda(step: int):
        if step < train_cfg.warmup_steps:
            return step / max(1, train_cfg.warmup_steps)

        progress = (step - train_cfg.warmup_steps) / max(
            1, train_cfg.total_steps - train_cfg.warmup_steps
        )
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # Trainer
    trainer = Trainer(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        tokenizer=tokenizer,
        train_cfg=train_cfg,
        device=device,
        logger=Logger(),
        scaler=scaler,
    )

    trainer.try_resume()
    trainer.train(loader)

    # Final save
    save_checkpoint(
        model=model,
        optimizer=optimizer,
        step=trainer.optimizer_step,
        scheduler=scheduler,
        path=f"releases/terry_tinyllm_{train_cfg.total_steps}.pt",
    )

    tokenizer.save_pretrained(train_cfg.tokenizer_dir)

    print("Training finished.")


class NotebookLogger(Logger):
    def __init__(self, window_size=100):
        super().__init__(window_size=window_size)
        self.history = {
            "step": [],
            "loss": [],
            "avg_loss": [],
            "total_avg_loss": [],
            "lr": [],
            "elapsed": [],
        }

    def log(self, step, loss, lr=None):
        self.losses.append(loss)
        self.total_steps += 1
        self.total_loss += loss

        elapsed = time.time() - self.start
        avg_loss = sum(self.losses) / len(self.losses)
        total_avg_loss = self.total_loss / self.total_steps

        self.history["step"].append(step)
        self.history["loss"].append(loss)
        self.history["avg_loss"].append(avg_loss)
        self.history["total_avg_loss"].append(total_avg_loss)
        self.history["lr"].append(lr if lr is not None else float("nan"))
        self.history["elapsed"].append(elapsed)

        lr_str = f" lr={lr:.2e}" if lr is not None else ""
        print(
            f"[{elapsed:7.1f}s] step={step:6d} "
            f"loss={loss:.4f} avg_loss={avg_loss:.4f} total_avg={total_avg_loss:.4f}{lr_str}"
        )


def plot_training_history(history):
    if not history["step"]:
        print("No training history recorded yet.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history["step"], history["loss"], marker="o", linewidth=1.5, label="loss")
    axes[0].plot(
        history["step"],
        history["avg_loss"],
        linewidth=2,
        label="running window avg",
    )
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Optimizer Step")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(history["step"], history["lr"], color="green", linewidth=2)
    axes[1].set_title("Learning Rate")
    axes[1].set_xlabel("Optimizer Step")
    axes[1].set_ylabel("LR")
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()


def run_training_notebook(
    model_cfg: ModelConfig | None = None,
    train_cfg: TrainConfig | None = None,
    resume: bool = True,
    plot: bool = True,
):
    model_cfg = model_cfg or ModelConfig()
    train_cfg = train_cfg or TrainConfig()

    set_seed(train_cfg.seed)
    device = setup_device(train_cfg.device)
    use_cuda = device.type == "cuda"

    tokenizer = load_tokenizer(train_cfg.tokenizer_dir)
    train_loader = build_dataloader(tokenizer, model_cfg, train_cfg, use_cuda)
    valid_loader = build_validation_dataloader(tokenizer, model_cfg, train_cfg, use_cuda)

    model = TinyLLM(model_cfg, vocab_size=len(tokenizer)).to(device)
    if model_cfg.max_seq_len > 4096:
        model.gradient_checkpointing_enable()

    scaler = None
    if train_cfg.use_mixed_precision and device.type == "cuda":
        scaler = torch.amp.GradScaler()
        print("Using mixed precision training")

    optimizer = AdamW(
        model.parameters(),
        lr=train_cfg.lr,
        weight_decay=train_cfg.weight_decay,
        fused=use_cuda,
    )

    def lr_lambda(step: int):
        if step < train_cfg.warmup_steps:
            return step / max(1, train_cfg.warmup_steps)
        progress = (step - train_cfg.warmup_steps) / max(
            1,
            train_cfg.total_steps - train_cfg.warmup_steps,
        )
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    logger = NotebookLogger()

    trainer = Trainer(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        tokenizer=tokenizer,
        train_cfg=train_cfg,
        device=device,
        logger=logger,
        scaler=scaler,
    )

    if resume:
        trainer.try_resume()

    trainer.train(train_loader)

    save_checkpoint(
        model=model,
        optimizer=optimizer,
        step=trainer.optimizer_step,
        scheduler=scheduler,
        path=f"releases/terry_tinyllm_{train_cfg.total_steps}.pt",
    )

    tokenizer.save_pretrained(train_cfg.tokenizer_dir)

    bundle = {
        "model": model,
        "tokenizer": tokenizer,
        "device": device,
        "model_cfg": model_cfg,
        "train_cfg": train_cfg,
        "train_loader": train_loader,
        "valid_loader": valid_loader,
        "trainer": trainer,
        "history": logger.history,
    }

    if plot and ENABLE_NOTEBOOK_PLOTS:
        plot_training_history(logger.history)

    print("Training finished.")
    return bundle


In [ ]:
# Cell 7: Validation (log and visual the status numbers in graphs)
def compute_masked_language_model_loss(logits, y, tokenizer, loss_mask=None):
    logits_flat = logits.reshape(-1, logits.size(-1))
    y_flat = y.reshape(-1)
    loss = F.cross_entropy(
        logits_flat,
        y_flat,
        ignore_index=tokenizer.pad_token_id,
        reduction="none",
    )

    if loss_mask is not None:
        mask_flat = loss_mask.reshape(-1).float()
        valid_flat = (y_flat != tokenizer.pad_token_id).float()
        effective_mask = mask_flat * valid_flat
        tokens = effective_mask.sum().item()
        if tokens == 0:
            return None, 0
        return (loss * effective_mask).sum().item() / tokens, int(tokens)

    valid_tokens = (y_flat != tokenizer.pad_token_id).sum().item()
    if valid_tokens == 0:
        return None, 0
    return loss.sum().item() / valid_tokens, int(valid_tokens)


@torch.no_grad()
def evaluate_validation_loader(
    model,
    dataloader,
    tokenizer,
    device,
    max_batches: int | None = None,
):
    was_training = model.training
    model.eval()

    history = {
        "batch": [],
        "loss": [],
        "running_loss": [],
        "tokens": [],
    }

    total_loss = 0.0
    total_tokens = 0

    for batch_idx, batch in enumerate(dataloader, start=1):
        if max_batches is not None and batch_idx > max_batches:
            break

        if len(batch) == 3:
            x, y, loss_mask = batch
            loss_mask = loss_mask.to(device)
        else:
            x, y = batch
            loss_mask = None

        x = x.to(device)
        y = y.to(device)
        padding_mask = x != tokenizer.pad_token_id
        logits = model(x, padding_mask=padding_mask)
        batch_loss, batch_tokens = compute_masked_language_model_loss(
            logits,
            y,
            tokenizer,
            loss_mask=loss_mask,
        )

        if batch_loss is None:
            continue

        total_loss += batch_loss * batch_tokens
        total_tokens += batch_tokens
        running_loss = total_loss / max(total_tokens, 1)

        history["batch"].append(batch_idx)
        history["loss"].append(batch_loss)
        history["running_loss"].append(running_loss)
        history["tokens"].append(batch_tokens)

        print(
            f"[valid] batch={batch_idx:4d} "
            f"loss={batch_loss:.4f} running_loss={running_loss:.4f} tokens={batch_tokens}"
        )

    if was_training:
        model.train()

    return history


def plot_validation_history(history):
    if not history["batch"]:
        print("No validation history recorded yet.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history["batch"], history["loss"], marker="o", linewidth=1.5)
    axes[0].set_title("Validation Loss Per Batch")
    axes[0].set_xlabel("Batch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history["batch"], history["running_loss"], color="orange", linewidth=2)
    axes[1].set_title("Validation Running Average Loss")
    axes[1].set_xlabel("Batch")
    axes[1].set_ylabel("Loss")
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()


def run_validation_notebook(
    training_bundle,
    max_batches: int | None = 50,
    plot: bool = True,
):
    model = training_bundle["model"]
    tokenizer = training_bundle["tokenizer"]
    device = training_bundle["device"]
    valid_loader = training_bundle.get("valid_loader")

    if valid_loader is None:
        model_cfg = training_bundle["model_cfg"]
        train_cfg = training_bundle["train_cfg"]
        use_cuda = device.type == "cuda"
        valid_loader = build_validation_dataloader(tokenizer, model_cfg, train_cfg, use_cuda)
        training_bundle["valid_loader"] = valid_loader

    history = evaluate_validation_loader(
        model=model,
        dataloader=valid_loader,
        tokenizer=tokenizer,
        device=device,
        max_batches=max_batches,
    )

    if plot and ENABLE_NOTEBOOK_PLOTS:
        plot_validation_history(history)

    training_bundle["validation_history"] = history
    return history


In [ ]:
# Cell 8: Inference tests

# Source: tools/inference.py
# Internal project imports were flattened into earlier notebook cells.

class ModelInference:
    """Load checkpoints and generate text continuations."""

    def __init__(
        self,
        checkpoint_path: str = "checkpoints/last_model.pt",
        device: str = "auto",
        tokenizer_path: str | None = None,
    ):
        self.model_config = ModelConfig()
        self.device = self._get_device(device)
        self.tokenizer = load_tokenizer(tokenizer_path)
        self._text_eos_token_ids = self.tokenizer.encode(
            self.tokenizer.eos_token,
            add_special_tokens=False,
        )
        self.model = self._load_model(checkpoint_path)

        print(f"Using device: {self.device}")
        print(f"Model loaded from: {checkpoint_path}")

    def _get_device(self, device: str) -> torch.device:
        if device == "auto":
            return torch.device("cuda" if torch.cuda.is_available() else "cpu")
        return torch.device(device)

    def _load_model(self, checkpoint_path: str) -> TinyLLM:
        checkpoint_file = Path(checkpoint_path)
        if not checkpoint_file.exists():
            raise FileNotFoundError(f"Checkpoint not found: {checkpoint_file}")

        checkpoint = torch.load(checkpoint_file, map_location=self.device)
        checkpoint_vocab_size = checkpoint["model"]["embed.weight"].size(0)
        tokenizer_vocab_size = len(self.tokenizer)
        if checkpoint_vocab_size != tokenizer_vocab_size:
            raise ValueError(
                "Checkpoint vocabulary does not match the Terry tokenizer. "
                f"checkpoint={checkpoint_vocab_size}, tokenizer={tokenizer_vocab_size}. "
                "Please retrain or point inference at a Terry-format checkpoint."
            )

        model = TinyLLM(
            self.model_config,
            vocab_size=len(self.tokenizer),
        ).to(self.device)

        model.load_state_dict(checkpoint["model"], strict=True)
        model.eval()

        if "step" in checkpoint:
            print(f"Checkpoint step: {checkpoint['step']}")

        return model

    def _build_input(self, prompt: str) -> torch.Tensor:
        input_ids = [self.tokenizer.bos_token_id]
        text = prompt.strip()
        if text:
            input_ids.extend(
                self.tokenizer.encode(
                    text,
                    add_special_tokens=False,
                )
            )
        return torch.tensor([input_ids], dtype=torch.long, device=self.device)

    def _decode_generated_reply(
        self,
        generated_ids: torch.Tensor,
        prompt_length: int,
    ) -> str:
        reply_ids = generated_ids[prompt_length:].tolist()

        if self.tokenizer.eos_token_id in reply_ids:
            stop = reply_ids.index(self.tokenizer.eos_token_id)
            reply_ids = reply_ids[:stop]

        text_eos_index = self._find_token_subsequence(
            reply_ids,
            self._text_eos_token_ids,
        )
        if text_eos_index != -1:
            reply_ids = reply_ids[:text_eos_index]

        return self.tokenizer.decode(
            reply_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        ).strip()

    @staticmethod
    def _find_token_subsequence(tokens: list[int], pattern: list[int]) -> int:
        if not pattern or len(pattern) > len(tokens):
            return -1

        last_start = len(tokens) - len(pattern) + 1
        for index in range(last_start):
            if tokens[index:index + len(pattern)] == pattern:
                return index

        return -1

    def generate(
        self,
        prompt: str,
        max_length: int = 200,
        temperature: float = 1.0,
        top_k: int = 50,
        top_p: float = 0.9,
        do_sample: bool = True,
    ) -> str:
        """Generate a continuation for a plain text prompt."""
        self.model.eval()
        input_ids = self._build_input(prompt)
        prompt_length = input_ids.size(1)

        with torch.no_grad():
            generated_ids = self.model.generate(
                input_ids,
                max_length=prompt_length + max_length,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                do_sample=do_sample,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        return self._decode_generated_reply(generated_ids[0], prompt_length)

    def generate_tokens(
        self,
        prompt: str,
        max_new_tokens: int = 50,
        temperature: float = 1.0,
        top_k: int = 50,
        top_p: float = 0.9,
        do_sample: bool = True,
    ) -> str:
        """Compatibility wrapper for older scripts."""
        return self.generate(
            prompt=prompt,
            max_length=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=do_sample,
        )

    def get_next_token_probabilities(
        self,
        prompt: str,
        top_k: int = 10,
        temperature: float = 1.0,
        top_p: float = 1.0,
    ):
        """Return the top token probabilities for the next token."""
        self.model.eval()

        input_ids = self._build_input(prompt)

        with torch.no_grad():
            padding_mask = input_ids != self.tokenizer.pad_token_id
            logits = self.model(input_ids, padding_mask=padding_mask)
            scale = temperature if temperature > 0 else 1.0
            next_logits = logits[:, -1, :] / scale

            if top_p is not None and top_p < 1.0:
                next_logits = self.model._top_p_filter(next_logits, top_p)

            if top_k is not None and top_k > 0:
                next_logits = self.model._top_k_filter(next_logits, top_k)

            probs = F.softmax(next_logits, dim=-1)[0]
            k = top_k if (top_k is not None and top_k > 0) else min(20, probs.size(0))
            values, indices = torch.topk(probs, k=k)

            results = []
            for token_id, probability in zip(indices.tolist(), values.tolist()):
                token_str = self.tokenizer.decode(
                    [token_id],
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True,
                )
                if not token_str and token_id == self.tokenizer.eos_token_id:
                    token_str = self.tokenizer.eos_token
                results.append((token_str, float(probability)))

        return results


def load_model(
    checkpoint_path: str = "checkpoints/last_model.pt",
    device: str = "auto",
    tokenizer_path: str | None = None,
):
    return ModelInference(checkpoint_path, device, tokenizer_path)

# Source: tools/checkpoint_manager.py
# Internal project imports were flattened into earlier notebook cells.

"""Checkpoint management utilities for TinyLLM."""






class CheckpointManager:
    def __init__(self, checkpoint_dir: str = "checkpoints", tokenizer_path: str | None = None):
        self.checkpoint_dir = checkpoint_dir
        self.tokenizer_path = tokenizer_path
        os.makedirs(checkpoint_dir, exist_ok=True)

    def list_checkpoints(self) -> list[dict[str, Any]]:
        """List all checkpoint files sorted by training step."""
        checkpoints: list[dict[str, Any]] = []

        if not os.path.exists(self.checkpoint_dir):
            return checkpoints

        for filename in os.listdir(self.checkpoint_dir):
            if not filename.endswith(".pt"):
                continue

            filepath = os.path.join(self.checkpoint_dir, filename)
            info = self.get_checkpoint_info(filepath)
            if info is None:
                continue

            info["filename"] = filename
            info["path"] = filepath
            checkpoints.append(info)

        checkpoints.sort(key=lambda item: item.get("step", 0), reverse=True)
        return checkpoints

    def get_checkpoint_info(self, checkpoint_path: str) -> dict[str, Any] | None:
        """Return metadata about a checkpoint without loading it onto GPU."""
        try:
            checkpoint = torch.load(checkpoint_path, map_location="cpu")
        except Exception as exc:
            print(f"Error reading checkpoint info from {checkpoint_path}: {exc}")
            return None

        return {
            "step": checkpoint.get("step", 0),
            "model_keys": list(checkpoint.get("model", {}).keys()),
            "optimizer_keys": list(checkpoint.get("optimizer", {}).keys()),
            "file_size": os.path.getsize(checkpoint_path),
            "file_path": checkpoint_path,
        }

    def load_model_from_checkpoint(
        self,
        checkpoint_path: str,
        device: str = "auto",
    ) -> TinyLLM:
        """Load a model from a checkpoint file."""
        if device == "auto":
            target_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            target_device = torch.device(device)

        tokenizer = load_tokenizer(self.tokenizer_path)
        checkpoint = torch.load(checkpoint_path, map_location=target_device)
        checkpoint_vocab_size = checkpoint["model"]["embed.weight"].size(0)
        tokenizer_vocab_size = len(tokenizer)
        if checkpoint_vocab_size != tokenizer_vocab_size:
            raise ValueError(
                "Checkpoint vocabulary does not match the Terry tokenizer. "
                f"checkpoint={checkpoint_vocab_size}, tokenizer={tokenizer_vocab_size}. "
                "Please retrain or load a Terry-format checkpoint."
            )

        model = TinyLLM(
            ModelConfig(),
            vocab_size=tokenizer_vocab_size,
        ).to(target_device)
        model.load_state_dict(checkpoint["model"])
        model.eval()
        return model

    def print_checkpoint_summary(self, checkpoint_path: str):
        """Print a short summary for a checkpoint file."""
        info = self.get_checkpoint_info(checkpoint_path)
        if not info:
            print(f"No checkpoint info available for {checkpoint_path}")
            return

        print("\n=== Checkpoint Summary ===")
        print(f"File: {os.path.basename(checkpoint_path)}")
        print(f"Path: {checkpoint_path}")
        print(f"Step: {info.get('step', 'N/A')}")
        print(f"File Size: {info.get('file_size', 0) / 1024 / 1024:.2f} MB")
        print(f"Model Keys: {len(info.get('model_keys', []))}")
        print(f"Optimizer Keys: {len(info.get('optimizer_keys', []))}")

    def get_latest_checkpoint(self) -> str | None:
        checkpoints = self.list_checkpoints()
        if checkpoints:
            return checkpoints[0]["path"]
        return None


def checkpoint_manager_main():
    manager = CheckpointManager()
    print("=== TinyLLM Checkpoint Manager ===")

    checkpoints = manager.list_checkpoints()
    if not checkpoints:
        print("No checkpoints found.")
        return

    print("\nAvailable checkpoints:")
    for index, checkpoint in enumerate(checkpoints, start=1):
        print(f"{index}. {checkpoint['filename']} (step: {checkpoint.get('step', 'N/A')})")

    latest = manager.get_latest_checkpoint()
    if latest:
        print(f"\nLatest checkpoint: {os.path.basename(latest)}")
        manager.print_checkpoint_summary(latest)


# Source: example_usage.py
# Internal project imports were flattened into earlier notebook cells.

"""Example script for chatting with Terry."""



def example_usage_main():
    print("=== Terry Inference Example ===\n")
    model = ModelInference(checkpoint_path="checkpoints/last_model.pt", device="auto")

    prompts = [
        "hi terry",
        "what do you think about rain on the window",
        "help me make a tiny plan for today",
        "tell me a small story",
    ]

    for index, prompt in enumerate(prompts, start=1):
        print(f"\n--- Prompt {index}: {prompt} ---")
        reply = model.generate(prompt, max_length=40, temperature=0.8)
        print(f"Terry: {reply}")

    print("\nEnter prompts to chat with Terry. Type 'quit' to exit.")
    while True:
        user_input = input("\nYou: ").strip()
        if user_input.lower() == "quit":
            print("Goodbye!")
            break
        if not user_input:
            continue

        try:
            reply = model.generate(user_input, max_length=60, temperature=0.8)
            print(f"Terry: {reply}")
        except Exception as exc:
            print(f"Error during generation: {exc}")


# Source: tests/test_train.py
# Internal project imports were flattened into earlier notebook cells.

class TestModelInternals(unittest.TestCase):
    def setUp(self):
        self.model_cfg = ModelConfig(
            d_model=64,
            n_layers=2,
            n_heads=2,
            max_seq_len=32,
            dropout_rate=0.0,
        )
        self.tokenizer = load_tokenizer()

    def test_01_vocab_resizing(self):
        """Resizing must preserve tied weights."""
        base_vocab_size = 128
        target_vocab_size = len(self.tokenizer)

        model = TinyLLM(self.model_cfg, vocab_size=base_vocab_size)
        model.resize_token_embeddings(target_vocab_size)

        self.assertEqual(model.embed.weight.size(0), target_vocab_size)
        self.assertEqual(model.head.weight.size(0), target_vocab_size)
        self.assertIs(model.head.weight, model.embed.weight)

    def test_02_serialize_text_record_mask_alignment(self):
        tokens, mask = serialize_text_record(self.tokenizer, "hello world")

        self.assertEqual(len(tokens), len(mask))
        self.assertEqual(tokens[0], self.tokenizer.bos_token_id)
        self.assertEqual(tokens[-1], self.tokenizer.eos_token_id)
        self.assertEqual(mask[0], 0)
        self.assertTrue(all(value == 1 for value in mask[1:]))


class TestTrainingLoop(unittest.TestCase):
    def setUp(self):
        self.model_cfg = ModelConfig(
            d_model=64,
            n_layers=2,
            n_heads=2,
            max_seq_len=32,
            dropout_rate=0.0,
        )
        self.tokenizer = load_tokenizer()
        self.vocab_size = len(self.tokenizer)

    def test_01_overfit_one_batch(self):
        """The model should overfit a single next-token batch quickly."""
        torch.manual_seed(42)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = TinyLLM(self.model_cfg, vocab_size=self.vocab_size).to(device)
        model.train()

        tokens = torch.randint(
            low=3,
            high=self.vocab_size,
            size=(2, self.model_cfg.max_seq_len + 1),
            device=device,
        )
        input_ids = tokens[:, :-1]
        labels = tokens[:, 1:]
        padding_mask = torch.ones_like(input_ids, dtype=torch.bool)

        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

        with torch.no_grad():
            initial_logits = model(input_ids, padding_mask=padding_mask)
            initial_loss = torch.nn.functional.cross_entropy(
                initial_logits.reshape(-1, self.vocab_size),
                labels.reshape(-1),
                ignore_index=self.tokenizer.pad_token_id,
            ).item()

        final_loss = initial_loss
        for _ in range(120):
            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids, padding_mask=padding_mask)
            loss = torch.nn.functional.cross_entropy(
                logits.reshape(-1, self.vocab_size),
                labels.reshape(-1),
                ignore_index=self.tokenizer.pad_token_id,
            )
            loss.backward()
            optimizer.step()
            final_loss = loss.item()

        self.assertLess(
            final_loss,
            initial_loss * 0.25,
            f"Expected strong overfitting, got loss {final_loss:.4f}",
        )


if __name__ == "__main__":
    unittest.main()

# Source: tests/test_inference.py
# Internal project imports were flattened into earlier notebook cells.

CHECKPOINT_PATH = "checkpoints/last_model.pt"


def checkpoint_is_compatible(path: str) -> bool:
    if not os.path.exists(path):
        return False

    checkpoint = torch.load(path, map_location="cpu")
    checkpoint_vocab_size = checkpoint["model"]["embed.weight"].size(0)
    tokenizer_vocab_size = len(load_tokenizer())
    return checkpoint_vocab_size == tokenizer_vocab_size


class TestTokenizer(unittest.TestCase):
    def test_round_trip(self):
        tokenizer = load_tokenizer()
        text = "hello owner"
        ids = tokenizer.encode(text)
        self.assertEqual(tokenizer.decode(ids), text)
        self.assertEqual(tokenizer.pad_token_id, 0)
        self.assertEqual(tokenizer.bos_token_id, 1)
        self.assertEqual(tokenizer.eos_token_id, 2)


class TestInferenceDecoding(unittest.TestCase):
    def test_decode_generated_reply_strips_literal_im_end_marker(self):
        tokenizer = load_tokenizer()
        model = ModelInference.__new__(ModelInference)
        model.tokenizer = tokenizer
        model._text_eos_token_ids = tokenizer.encode(
            tokenizer.eos_token,
            add_special_tokens=False,
        )

        prompt_length = 4
        reply_ids = tokenizer.encode(
            "hello owner\n<|im_end|>\nextra text",
            add_special_tokens=False,
        )
        generated_ids = torch.tensor([11, 12, 13, 14, *reply_ids], dtype=torch.long)

        reply = model._decode_generated_reply(generated_ids, prompt_length)
        self.assertEqual(reply, "hello owner")


@unittest.skipUnless(
    checkpoint_is_compatible(CHECKPOINT_PATH),
    "compatible Terry checkpoint required for inference smoke tests",
)
class TestInference(unittest.TestCase):
    def test_model_loading(self):
        model = ModelInference(checkpoint_path=CHECKPOINT_PATH)
        self.assertFalse(model.model.training)

    def test_generation(self):
        model = ModelInference(checkpoint_path=CHECKPOINT_PATH)
        reply = model.generate("hi terry", max_length=16, do_sample=False, temperature=0.0)
        self.assertIsInstance(reply, str)

    def test_checkpoint_manager(self):
        manager = CheckpointManager()
        latest = manager.get_latest_checkpoint()
        self.assertIsNotNone(latest)
        model = manager.load_model_from_checkpoint(latest)
        self.assertIsNotNone(model)


if __name__ == "__main__":
    unittest.main()

# Source: tests/test_variable_length.py
# Internal project imports were flattened into earlier notebook cells.

"""Quick smoke test for variable-length sequence support."""






def test_variable_length_sequences():
    """Test that the dataset and model work with dynamic padding."""
    print("Testing variable-length sequence support...")

    tokenizer = load_tokenizer()

    documents = [
        [10, 11, 12, 13, 14, 15, 16, tokenizer.eos_token_id],
        [20, 21, 22, 23, tokenizer.eos_token_id],
        [30, 31, 32, 33, 34, 35, 36, 37, 38, tokenizer.eos_token_id],
        [40, 41, 42, tokenizer.eos_token_id],
    ]

    max_seq_len = 8
    dataset = TokenDataset(documents, max_seq_len, min_seq_len=2)

    print(f"Dataset length: {len(dataset)}")

    def collate_fn(batch):
        input_seqs, target_seqs = zip(*batch)
        max_len = max(seq.size(0) for seq in input_seqs)

        batch_inputs = torch.stack(
            [
                torch.nn.functional.pad(
                    seq,
                    (0, max_len - seq.size(0)),
                    value=tokenizer.pad_token_id,
                )
                for seq in input_seqs
            ]
        )
        batch_targets = torch.stack(
            [
                torch.nn.functional.pad(
                    seq,
                    (0, max_len - seq.size(0)),
                    value=tokenizer.pad_token_id,
                )
                for seq in target_seqs
            ]
        )
        return batch_inputs, batch_targets

    batch = [dataset[i] for i in range(min(3, len(dataset)))]
    batch_inputs, batch_targets = collate_fn(batch)
    padding_mask = batch_inputs != tokenizer.pad_token_id

    model_cfg = ModelConfig(max_seq_len=max_seq_len)
    model = TinyLLM(model_cfg, vocab_size=len(tokenizer))
    model.eval()

    with torch.no_grad():
        logits = model(batch_inputs, padding_mask=padding_mask)

    print(f"Batch shape: {batch_inputs.shape}")
    print(f"Target shape: {batch_targets.shape}")
    print(f"Model output shape: {logits.shape}")
    print(f"Causal mask cached: {model.blocks[0].attn.causal_mask is not None}")
    print("Variable-length sequence test passed.")


def run_notebook_inference_demo(
    prompt: str = "hi terry",
    checkpoint_path: str = "checkpoints/last_model.pt",
    device: str = "auto",
    max_length: int = 40,
    temperature: float = 0.8,
):
    inference = ModelInference(
        checkpoint_path=checkpoint_path,
        device=device,
    )
    reply = inference.generate(
        prompt,
        max_length=max_length,
        temperature=temperature,
    )
    print(f"Prompt: {prompt}")
    print(f"Terry: {reply}")
    return reply


def run_notebook_smoke_tests(verbosity: int = 2):
    suite = unittest.TestSuite()
    loader = unittest.defaultTestLoader

    for case in (
        TestTokenizer,
        TestModelInternals,
        TestTrainingLoop,
        TestInference,
    ):
        suite.addTests(loader.loadTestsFromTestCase(case))

    suite.addTests(loader.loadTestsFromName("test_variable_length_sequences"))
    return unittest.TextTestRunner(verbosity=verbosity).run(suite)


In [ ]:
train_cfg = TrainConfig(total_steps=1000)
training_bundle = run_training_notebook(train_cfg=train_cfg)

In [ ]:
validation_history = run_validation_notebook(training_bundle, max_batches=50)

In [ ]:
run_notebook_inference_demo(
    prompt="hi terry",
    checkpoint_path="checkpoints/last_model.pt",
)